# Vibes Audit Pipeline

Audit propensity evaluation results from `niels/propensities/results/`.

Steps:
1. Setup & install deps
2. Discover available results
3. Generate audit configs
4. Stratified sampling
5. Run alternative judges
6. Human annotation (optional)
7. Analysis & summary

## 1. Setup

In [3]:
!pip install -q pyyaml pandas numpy scipy scikit-learn tenacity tqdm openai anthropic python-dotenv

In [4]:
import os
from dotenv import load_dotenv
from google.colab import drive, userdata

# Mount Google Drive (optional, if needed for other files)
drive.mount('/content/drive')

# Set API keys for alt judges using Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get("openai")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("anthropic")

# Or load from .env if available (this will be overridden by Colab secrets if keys are present there)
load_dotenv()

Mounted at /content/drive


False

In [5]:
from bridge import discover_results, load_results, list_available_evals, get_eval_info
from generate_configs import generate_and_save, CONFIGS_DIR
from audit_config import from_yaml
from sample_for_review import load_data, stratified_sample
from run_alt_judges import run_judges
from analyze import inter_judge_correlations, bias_probes, audit_summary
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("output")

# Which eval to audit — change this to audit a different propensity
EVAL_NAME = "risk_affinity"

print("Available evals:", list_available_evals())

ModuleNotFoundError: No module named 'bridge'

## 2. Discover Results

In [ ]:
info = get_eval_info(EVAL_NAME)
print(f"Eval: {EVAL_NAME}")
print(f"Metrics: {info['judge_metrics']}")
print(f"YAML: {info['yaml_path']}")
print(f"Result CSVs: {len(info['results_csvs'])}")

for csv_path in info["results_csvs"]:
    df = pd.read_csv(csv_path, low_memory=False)
    models = df["model"].unique() if "model" in df.columns else ["unknown"]
    elicitations = df["elicitation"].unique() if "elicitation" in df.columns else ["unknown"]
    print(f"  {csv_path.name}: {len(df)} rows, models={list(models)}, elicitations={list(elicitations)}")

## 3. Generate Audit Config

In [ ]:
metric = info["judge_metrics"][0]
print(f"Primary metric: {metric}")

config_path = generate_and_save(EVAL_NAME, metric)

# Show the generated config
print(f"\nGenerated config at {config_path}:")
print(open(config_path).read())

## 4. Load Results & Stratified Sampling

In [ ]:
# Load all results for this eval
results_df = load_results(EVAL_NAME)
print(f"Total rows: {len(results_df)}")
print(f"Models: {results_df['model'].unique().tolist()}")
print(f"Elicitations: {results_df['elicitation'].unique().tolist()}")
print(f"\nScore stats ({metric}):")
print(results_df[metric].describe())

In [ ]:
# Save consolidated results and run stratified sampling
eval_output = OUTPUT_DIR / EVAL_NAME
eval_output.mkdir(parents=True, exist_ok=True)

data_path = eval_output / "all_results.csv"
results_df.to_csv(data_path, index=False)

config = from_yaml(config_path, data_path=str(data_path), output_dir=str(eval_output))
TARGET_N = 200
config.target_n = min(TARGET_N, len(results_df))

df_loaded = load_data(config)
sample = stratified_sample(df_loaded, config)

print(f"\nSampled: {len(sample)} rows")

# Save full sample
full_path = eval_output / f"sample_{len(sample)}.csv"
sample.to_csv(full_path, index=False)
print(f"Saved: {full_path}")

# Save blind sample (for human annotation)
blind_cols = ["question", "response"] + [
    c for c in config.metadata_columns if c in sample.columns
]
blind = sample[blind_cols].copy()
blind["human_label"] = ""
blind_path = eval_output / f"sample_{len(sample)}_blind.csv"
blind.to_csv(blind_path, index=False)
print(f"Saved blind: {blind_path}")

In [ ]:
# Preview the sample
sample[["question", "response", metric, "model", "elicitation"]].head()

## 5. Run Alternative Judges

Requires `OPENAI_API_KEY` and/or `ANTHROPIC_API_KEY` set above.

In [ ]:
sample_df = pd.read_csv(full_path, low_memory=False)
config = from_yaml(config_path, output_dir=str(eval_output))

print(f"Running alt judges on {len(sample_df)} rows...")
print(f"Judges: {[j['name'] for j in config.alt_judges]}")

result = run_judges(sample_df, config)

alt_path = eval_output / "alt_judge_scores.csv"
result.to_csv(alt_path, index=False)
print(f"\nSaved: {alt_path}")

In [ ]:
# Quick correlation check
score_cols = [c for c in result.columns if c.endswith("_score") and c != config.score_column]
for col in score_cols:
    valid = result[col].notna() & result[config.score_column].notna()
    if valid.sum() > 0:
        corr = result.loc[valid, col].corr(result.loc[valid, config.score_column])
        print(f"Correlation {config.score_column} vs {col}: {corr:.3f}")

## 6. Human Annotation (Optional)

The annotation GUI requires a local server. In Colab, you can review samples manually instead.

To use the full GUI locally:
```bash
cd june/vibes_audit
python annotate.py --config configs/<eval>__<metric>.yaml --output-dir output/<eval>
```

In [ ]:
# Manual review: inspect a few samples
blind_df = pd.read_csv(blind_path)
for i, row in blind_df.head(5).iterrows():
    print(f"\n{'='*60}")
    print(f"Sample {i} | model={row.get('model', '?')} | elicitation={row.get('elicitation', '?')}")
    print(f"{'='*60}")
    print(f"Q: {row['question'][:200]}...")
    print(f"\nA: {row['response'][:300]}...")

## 6b. Load Human Annotations

After running the annotation GUI locally, upload or load `human_annotations.csv`.

In [ ]:
import numpy as np
from analyze import gwets_ac2, cohen_weighted_kappa, score_to_bins, confusion_matrix_plot

ann_path = eval_output / "human_annotations.csv"

# In Colab, upload the file if running remotely:
# from google.colab import files
# uploaded = files.upload()  # upload human_annotations.csv
# import shutil; shutil.move("human_annotations.csv", str(ann_path))

if not ann_path.exists():
    print(f"No annotations found at {ann_path}")
    print("Run the annotator locally first:")
    print(f"  python annotate.py --config configs/{EVAL_NAME}__{metric}.yaml --output-dir output/{EVAL_NAME}")
else:
    human_df = pd.read_csv(ann_path)
    n_labeled = human_df["human_label"].notna() & (human_df["human_label"] != "")
    print(f"Loaded {n_labeled.sum()} / {len(human_df)} annotations from {ann_path}")
    print(f"\nLabel distribution:")
    print(human_df["human_label"].value_counts())

### Human vs Judge Agreement

Compare human labels against the original judge scores and alt judge scores.

In [ ]:
if ann_path.exists():
    import matplotlib.pyplot as plt

    human_df = pd.read_csv(ann_path)
    config = from_yaml(config_path, output_dir=str(eval_output))

    # Map human labels to numeric (bucket number)
    label_to_num = {b.label: b.number for b in config.buckets}
    label_to_num["INCOHERENT"] = 0

    labeled = human_df[human_df["human_label"].notna() & (human_df["human_label"] != "")].copy()
    labeled["human_score"] = labeled["human_label"].map(label_to_num)
    print(f"Analyzing {len(labeled)} labeled samples\n")

    # Merge with full sample to get judge scores
    sample_df = pd.read_csv(full_path, low_memory=False)
    # Align by index position (annotator uses row index)
    if "index" in labeled.columns:
        labeled = labeled.set_index("index")
    merged = sample_df.loc[labeled.index].copy()
    merged["human_label"] = labeled["human_label"].values
    merged["human_score"] = labeled["human_score"].values

    # Filter out INCOHERENT for numeric comparisons
    valid = merged[merged["human_score"] > 0].copy()
    print(f"Valid for numeric comparison: {len(valid)} (excluded {len(merged) - len(valid)} INCOHERENT)\n")

    # Bin judge scores to same 5-bucket scale
    bucket_edges = [0, 20, 40, 60, 80, 100]
    bucket_labels = [1, 2, 3, 4, 5]  # Very Low to Very High
    valid["judge_bucket"] = pd.cut(
        valid[metric], bins=bucket_edges, labels=bucket_labels, include_lowest=True
    ).astype(int)

    # --- Agreement metrics ---
    print("=" * 60)
    print("HUMAN vs ORIGINAL JUDGE AGREEMENT")
    print("=" * 60)

    # Gwet's AC2
    ac2 = gwets_ac2(valid["human_score"].tolist(), valid["judge_bucket"].tolist(), bucket_labels)
    print(f"  Gwet's AC2:             {ac2:.3f}")

    # Weighted Kappa
    wk = cohen_weighted_kappa(valid["human_score"].values, valid["judge_bucket"].values, 5)
    print(f"  Weighted Cohen's Kappa: {wk:.3f}")

    # Exact match
    exact = (valid["human_score"] == valid["judge_bucket"]).mean()
    print(f"  Exact match:            {exact:.1%}")

    # Within-1 match
    within1 = (abs(valid["human_score"] - valid["judge_bucket"]) <= 1).mean()
    print(f"  Within 1 bucket:        {within1:.1%}")

    # Correlation: human bucket vs raw judge score
    from scipy import stats
    r, p = stats.spearmanr(valid["human_score"], valid[metric])
    print(f"  Spearman (human vs raw): r={r:.3f}, p={p:.4f}")

    # --- Confusion matrix ---
    bucket_names = [b.label for b in reversed(config.buckets)]  # VL to VH
    fig, ax = plt.subplots(figsize=(7, 6))
    confusion_matrix_plot(
        valid["judge_bucket"].values,
        valid["human_score"].values,
        labels=bucket_labels,
        title=f"{EVAL_NAME}: Judge Bucket vs Human Label",
        ax=ax,
    )
    ax.set_xlabel("Human Label (bucket)")
    ax.set_ylabel("Judge Score (binned)")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping — no human annotations found.")

In [ ]:
# Disagreement analysis: which samples did human and judge disagree on most?
if ann_path.exists() and len(valid) > 0:
    valid["disagreement"] = abs(valid["human_score"] - valid["judge_bucket"])
    disagreed = valid[valid["disagreement"] >= 2].sort_values("disagreement", ascending=False)

    print(f"Large disagreements (>= 2 buckets apart): {len(disagreed)} / {len(valid)}")
    print()

    for i, (_, row) in enumerate(disagreed.head(10).iterrows()):
        print(f"--- Disagreement #{i+1}: human={int(row['human_score'])} vs judge_bucket={int(row['judge_bucket'])} (raw={row[metric]:.0f}) ---")
        print(f"  Model: {row.get('model', '?')} | Elicitation: {row.get('elicitation', '?')}")
        print(f"  Q: {str(row['question'])[:150]}...")
        print(f"  A: {str(row['response'])[:200]}...")
        print()

### Systematic Bias Analysis

Quantify the direction and source of disagreements between human and judge.

In [ ]:
if ann_path.exists() and len(valid) > 0:
    import matplotlib.pyplot as plt
    from scipy import stats

    # --- Directional bias ---
    valid["delta"] = valid["human_score"] - valid["judge_bucket"]  # positive = judge too low
    mean_delta = valid["delta"].mean()
    judge_too_low = (valid["delta"] > 0).sum()
    judge_too_high = (valid["delta"] < 0).sum()
    exact_agree = (valid["delta"] == 0).sum()

    print("=" * 60)
    print("DIRECTIONAL BIAS (human - judge_bucket)")
    print("=" * 60)
    print(f"  Mean delta:       {mean_delta:+.2f} buckets {'(judge scores too LOW)' if mean_delta > 0 else '(judge scores too HIGH)'}")
    print(f"  Judge too low:    {judge_too_low} ({judge_too_low/len(valid):.0%})")
    print(f"  Exact agreement:  {exact_agree} ({exact_agree/len(valid):.0%})")
    print(f"  Judge too high:   {judge_too_high} ({judge_too_high/len(valid):.0%})")

    # Wilcoxon signed-rank test for systematic bias
    if len(valid[valid["delta"] != 0]) >= 10:
        stat, p = stats.wilcoxon(valid["delta"])
        print(f"\n  Wilcoxon signed-rank: W={stat:.0f}, p={p:.4f}")
        print(f"  {'Significant systematic bias' if p < 0.05 else 'No significant systematic bias'} (alpha=0.05)")

    # --- Distribution of deltas ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram of deltas
    ax = axes[0]
    delta_counts = valid["delta"].value_counts().sort_index()
    colors = ["#da3633" if d < 0 else "#238636" if d > 0 else "#8b949e" for d in delta_counts.index]
    ax.bar(delta_counts.index, delta_counts.values, color=colors, edgecolor="white", linewidth=0.5)
    ax.axvline(0, color="white", linewidth=1, linestyle="--", alpha=0.5)
    ax.set_xlabel("Human - Judge (buckets)")
    ax.set_ylabel("Count")
    ax.set_title("Disagreement Direction\n(positive = judge too low, negative = judge too high)")
    ax.set_xticks(range(int(valid["delta"].min()), int(valid["delta"].max()) + 1))

    # Scatter: raw judge score vs human bucket
    ax = axes[1]
    jitter = np.random.RandomState(42).uniform(-0.15, 0.15, len(valid))
    ax.scatter(valid["human_score"] + jitter, valid[metric], alpha=0.5, s=30, c="#58a6ff", edgecolors="none")
    # Add bucket boundary lines
    for edge in bucket_edges[1:-1]:
        ax.axhline(edge, color="#30363d", linewidth=0.8, linestyle="--")
    ax.set_xlabel("Human Label (bucket)")
    ax.set_ylabel(f"Judge Raw Score ({metric})")
    ax.set_title("Human Label vs Raw Judge Score")
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_xticklabels(["Very\nLow", "Low", "Mod", "High", "Very\nHigh"], fontsize=9)

    plt.tight_layout()
    plt.show()

In [ ]:
if ann_path.exists() and len(valid) > 0:
    # --- Breakdown by model ---
    print("=" * 60)
    print("DISAGREEMENT BY MODEL")
    print("=" * 60)

    model_stats = valid.groupby("model").agg(
        n=("delta", "size"),
        mean_delta=("delta", "mean"),
        large_disagree=("disagreement", lambda x: (x >= 2).sum()),
        mean_judge=( metric, "mean"),
        mean_human=("human_score", "mean"),
    ).sort_values("mean_delta", ascending=False)
    model_stats["large_disagree_pct"] = (model_stats["large_disagree"] / model_stats["n"] * 100).round(1)
    model_stats["mean_delta"] = model_stats["mean_delta"].round(2)
    model_stats["mean_judge"] = model_stats["mean_judge"].round(1)
    model_stats["mean_human"] = model_stats["mean_human"].round(2)

    print(model_stats.to_string())

    # --- Breakdown by elicitation ---
    print(f"\n{'=' * 60}")
    print("DISAGREEMENT BY ELICITATION")
    print("=" * 60)

    elic_stats = valid.groupby("elicitation").agg(
        n=("delta", "size"),
        mean_delta=("delta", "mean"),
        large_disagree=("disagreement", lambda x: (x >= 2).sum()),
        mean_judge=(metric, "mean"),
        mean_human=("human_score", "mean"),
    ).sort_values("mean_delta", ascending=False)
    elic_stats["large_disagree_pct"] = (elic_stats["large_disagree"] / elic_stats["n"] * 100).round(1)
    elic_stats["mean_delta"] = elic_stats["mean_delta"].round(2)
    elic_stats["mean_judge"] = elic_stats["mean_judge"].round(1)
    elic_stats["mean_human"] = elic_stats["mean_human"].round(2)

    print(elic_stats.to_string())

    # --- Plot ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # By model
    ax = axes[0]
    models_sorted = model_stats.sort_values("mean_delta")
    colors = ["#da3633" if d < -0.3 else "#238636" if d > 0.3 else "#9e6a03" for d in models_sorted["mean_delta"]]
    bars = ax.barh(range(len(models_sorted)), models_sorted["mean_delta"], color=colors)
    ax.set_yticks(range(len(models_sorted)))
    ax.set_yticklabels([m.split("/")[-1] if "/" in m else m for m in models_sorted.index], fontsize=9)
    ax.axvline(0, color="white", linewidth=0.8)
    ax.set_xlabel("Mean Delta (human - judge bucket)")
    ax.set_title("Judge Bias by Model\n(positive = judge too low)")

    # By elicitation
    ax = axes[1]
    elic_sorted = elic_stats.sort_values("mean_delta")
    colors = ["#da3633" if d < -0.3 else "#238636" if d > 0.3 else "#9e6a03" for d in elic_sorted["mean_delta"]]
    ax.barh(range(len(elic_sorted)), elic_sorted["mean_delta"], color=colors)
    ax.set_yticks(range(len(elic_sorted)))
    ax.set_yticklabels(elic_sorted.index, fontsize=9)
    ax.axvline(0, color="white", linewidth=0.8)
    ax.set_xlabel("Mean Delta (human - judge bucket)")
    ax.set_title("Judge Bias by Elicitation\n(positive = judge too low)")

    plt.tight_layout()
    plt.show()

In [ ]:
if ann_path.exists() and len(valid) > 0:
    # --- Honest-refusal bias probe ---
    # Detect responses with capability disclaimers / honest refusals
    # that the judge may be penalizing as "risk-averse"
    refusal_patterns = [
        r"I can(?:'t|not)\b",
        r"I(?:'m| am) (?:a text|an AI|not able)",
        r"cannot (?:produce|generate|create|attach|provide files)",
        r"beyond (?:my|what I can)",
    ]
    import re
    pattern = "|".join(refusal_patterns)
    valid["has_refusal"] = valid["response"].astype(str).apply(
        lambda x: bool(re.search(pattern, x, re.IGNORECASE))
    )

    refusal_rows = valid[valid["has_refusal"]]
    non_refusal = valid[~valid["has_refusal"]]

    print("=" * 60)
    print("HONEST-REFUSAL BIAS PROBE")
    print("=" * 60)
    print(f"  Responses with capability disclaimers: {len(refusal_rows)} / {len(valid)} ({len(refusal_rows)/len(valid):.0%})")
    print()

    if len(refusal_rows) >= 3 and len(non_refusal) >= 3:
        print(f"  {'':30s} {'With refusal':>14s} {'Without':>14s}")
        print(f"  {'Mean judge raw score':30s} {refusal_rows[metric].mean():14.1f} {non_refusal[metric].mean():14.1f}")
        print(f"  {'Mean human bucket':30s} {refusal_rows['human_score'].mean():14.2f} {non_refusal['human_score'].mean():14.2f}")
        print(f"  {'Mean delta (human - judge)':30s} {refusal_rows['delta'].mean():+14.2f} {non_refusal['delta'].mean():+14.2f}")
        print(f"  {'Large disagreements':30s} {(refusal_rows['disagreement'] >= 2).sum():14d} {(non_refusal['disagreement'] >= 2).sum():14d}")
        print(f"  {'Large disagree %':30s} {(refusal_rows['disagreement'] >= 2).mean()*100:13.1f}% {(non_refusal['disagreement'] >= 2).mean()*100:13.1f}%")

        # Mann-Whitney test
        u_stat, u_p = stats.mannwhitneyu(
            refusal_rows["delta"], non_refusal["delta"], alternative="greater"
        )
        print(f"\n  Mann-Whitney U (refusal delta > non-refusal delta): U={u_stat:.0f}, p={u_p:.4f}")
        print(f"  {'Significant' if u_p < 0.05 else 'Not significant'}: judge penalizes honest-refusal responses {'MORE' if u_p < 0.05 else 'similarly'}")
    else:
        print("  Not enough samples in each group for comparison.")

## 7. Analysis & Summary

In [ ]:
alt_df = pd.read_csv(alt_path, low_memory=False)
config = from_yaml(config_path, output_dir=str(eval_output))

# Check for human annotations
ann_path = eval_output / "human_annotations.csv"
human_df = pd.read_csv(ann_path) if ann_path.exists() else pd.DataFrame()

# Audit summary
summary = audit_summary(config, human_df, alt_df)
summary["eval"] = EVAL_NAME

summary_path = eval_output / "audit_summary.csv"
summary.to_csv(summary_path, index=False)

print(f"{EVAL_NAME} Audit Summary:")
print(f"{'='*70}")
for _, row in summary.iterrows():
    icon = {"PASS": "PASS", "MARGINAL": "WARN", "FAIL": "FAIL"}.get(row["Status"], "?")
    print(f"  [{icon:>4}] {row['Metric']}: {row['Value']} (threshold {row['Threshold']})")

summary

In [ ]:
# Bias probes
group_cols = [c for c in config.metadata_columns if c in alt_df.columns]
probes = bias_probes(alt_df, config.score_column, group_cols)

print("Bias Probes:")
print(f"{'='*70}")
for probe_name, res in probes.items():
    if "r" in res:
        print(f"  {probe_name}: r={res['r']:.3f}, p={res['p']:.4f}")
    elif "F" in res:
        print(f"  {probe_name}: F={res['F']:.2f}, p={res['p']:.4f}")
        if "group_means" in res:
            for gname, gmean in res["group_means"].items():
                print(f"    {gname}: mean={gmean:.1f}")

In [ ]:
# Inter-judge correlations
all_score_cols = [config.score_column] + [
    c for c in alt_df.columns if c.endswith("_score") and c != config.score_column
]
corr_df = inter_judge_correlations(alt_df, all_score_cols)
print("Inter-Judge Correlations:")
corr_df